# Llama 3.1 8B — LoRA 병합 → GGUF Q3_K_M (Kaggle P100)

| 단계 | 작업 | 소요 시간 |
|------|------|-----------|
| 1 | LoRA 병합 (FP16, GPU+CPU 분산) | ~10분 |
| 2 | llama.cpp 빌드 | ~5분 |
| 3 | GGUF F16 변환 | ~5분 |
| 4 | Q3_K_M 양자화 | ~5분 |
| 5 | 결과 다운로드 (~3.3GB) | 직접 다운로드 |

**실행 전 체크리스트**
1. 오른쪽 패널 → **Accelerator: GPU P100** 선택
2. `Add-ons` → `Secrets` → `HF_TOKEN` 등록 후 노트북에서 활성화
3. `Add-ons` → `Datasets` → adapters_v3 데이터셋 추가
4. 셀 순서대로 실행

In [ ]:
# 1. GPU + 디스크 공간 확인
import torch, shutil
assert torch.cuda.is_available(), 'GPU 없음. Accelerator를 P100으로 변경하세요.'
print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
total, used, free = shutil.disk_usage('/')
print(f'Disk : {free / 1e9:.1f} GB 여유 (최소 40GB 필요)')
if free / 1e9 < 40:
    print('  ⚠ 공간 부족')
else:
    print('  ✓ OK')

In [ ]:
# 2. 패키지 설치
!pip install -q "transformers>=4.45.0" "peft>=0.13.0" "accelerate>=0.33.0" "bitsandbytes>=0.43.0"
print('설치 완료')

In [ ]:
# 3. HuggingFace 인증
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

try:
    token = UserSecretsClient().get_secret('HF_TOKEN')
    login(token=token, add_to_git_credential=False)
    print('HF_TOKEN 로드 성공')
except Exception:
    print('Secrets에 HF_TOKEN 없음 → 수동 입력')
    login()

In [ ]:
# 4. 경로 설정 (Kaggle)
import os, glob

BASE_MODEL  = 'meta-llama/Meta-Llama-3.1-8B-Instruct'
MERGED_DIR  = '/kaggle/working/merged_model_v3'
GGUF_F16    = '/kaggle/working/driving-mentor-v3-f16.gguf'
GGUF_Q3KM   = '/kaggle/working/driving-mentor-v3-q3km.gguf'

# 데이터셋 경로 자동 탐색 (Add-ons → Datasets에 추가한 adapters_v3)
candidates = glob.glob('/kaggle/input/**/adapter_config.json', recursive=True)
assert candidates, '어댑터 데이터셋을 찾을 수 없습니다. Add-ons → Datasets에서 adapters_v3를 추가하세요.'
ADAPTER_DIR = os.path.dirname(candidates[0])
print(f'어댑터 경로: {ADAPTER_DIR}')
print(f'어댑터 파일 목록: {os.listdir(ADAPTER_DIR)}')

In [ ]:
# 5. LoRA 어댑터 병합 → FP16 모델 저장
# P100 CUDA sm_60 → 현재 PyTorch(sm_70 이상 필요) 비호환
# 전략: CPU 전용 로드 (FP16, 30GB RAM 사용) → merge_and_unload → 저장
import gc, torch, os
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

print('[1/3] 베이스 모델 로드 (FP16, CPU 전용)...')
print('  ※ P100 sm_60 PyTorch 비호환으로 CPU 사용 (~20분 소요)')
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map='cpu',
    trust_remote_code=True,
)
print('  로드 완료')

print('[2/3] 어댑터 로드 및 병합 (merge_and_unload)...')
model = PeftModel.from_pretrained(model, ADAPTER_DIR)
model = model.merge_and_unload()
print('  병합 완료')

print('[3/3] 병합 모델 저장 (2GB 샤드)...')
os.makedirs(MERGED_DIR, exist_ok=True)
model.save_pretrained(MERGED_DIR, safe_serialization=True, max_shard_size='2GB')
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True)
tokenizer.save_pretrained(MERGED_DIR)

del model
gc.collect()
print(f'저장 완료: {os.listdir(MERGED_DIR)}')

In [ ]:
# 6. llama.cpp 클론 + llama-quantize 빌드 (~5분)
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /kaggle/working/llama.cpp 2>/dev/null || echo '이미 클론됨'
!pip install -q -r /kaggle/working/llama.cpp/requirements.txt
!cmake -B /kaggle/working/llama.cpp/build -S /kaggle/working/llama.cpp -DGGML_CUDA=OFF -DCMAKE_BUILD_TYPE=Release -DBUILD_SHARED_LIBS=OFF > /dev/null 2>&1
!cmake --build /kaggle/working/llama.cpp/build --target llama-quantize -j$(nproc) 2>&1 | tail -3

import glob
bins = glob.glob('/kaggle/working/llama.cpp/build/**/llama-quantize', recursive=True)
if not bins:
    bins = glob.glob('/kaggle/working/llama.cpp/build/**/quantize', recursive=True)
assert bins, 'llama-quantize 빌드 실패 — 위 출력을 확인하세요'
QUANTIZE_BIN = bins[0]
os.chmod(QUANTIZE_BIN, 0o755)
print(f'빌드 성공: {QUANTIZE_BIN}')

In [ ]:
# 6-5. 디스크 공간 확보 (output 19.5GB 한계 대응)
# merged_model_v3(16GB)을 /tmp로 이동 → /kaggle/working 확보
import shutil, os

!rm -f {GGUF_F16}
print('불완전한 F16 GGUF 삭제')

!mv {MERGED_DIR} /tmp/merged_model_v3
MERGED_DIR = '/tmp/merged_model_v3'
print(f'merged_model_v3 → /tmp 이동 완료')

total, used, free = shutil.disk_usage('/kaggle/working')
print(f'/kaggle/working 여유: {free/1e9:.1f} GB')

In [ ]:
# 7. 병합 모델 → GGUF F16 변환 (~5분)
# F16을 /tmp에 저장 (root overlay 1.1TB 여유, /kaggle/working 20GB 제한 회피)
import glob, os

GGUF_F16_TMP = '/tmp/driving-mentor-v3-f16.gguf'
CONVERT_SCRIPT = '/kaggle/working/llama.cpp/convert_hf_to_gguf.py'
assert os.path.exists(CONVERT_SCRIPT), 'convert script 없음'

print('[1/2] F16 변환 → /tmp (~5분)...')
!python {CONVERT_SCRIPT} {MERGED_DIR} --outtype f16 --outfile {GGUF_F16_TMP}
!ls -lh {GGUF_F16_TMP}

In [ ]:
# 8. GGUF F16 → Q3_K_M 양자화 (~5분)
# Q8_0 → Q3_K_M은 llama.cpp에서 비활성화됨 → F16에서 직접 양자화
# Q3_K_M: ~3.3GB (Q4_K_M 4.5GB 대비 1.2GB 절약 → RTX 2060 6GB 100% GPU 가능)
print('[2/2] Q3_K_M 양자화...')
!{QUANTIZE_BIN} {GGUF_F16_TMP} {GGUF_Q3KM} Q3_K_M
!ls -lh {GGUF_Q3KM}
!rm -f {GGUF_F16_TMP}
print('F16 중간 파일 삭제 완료')

In [ ]:
# 9. 완료 확인 + 다운로드 안내
import os
size = os.path.getsize(GGUF_Q3KM) / 1e9
print(f'완료: {GGUF_Q3KM}  ({size:.2f} GB)')
print()
print('다운로드 방법:')
print('  오른쪽 패널 Output 탭 → driving-mentor-v3-q3km.gguf → 다운로드 버튼')

## 로컬 PC에서 Ollama 등록

Kaggle Output 탭에서 `driving-mentor-v3-q3km.gguf` 다운로드 후 `LCAE-AI/` 폴더에 놓고:

```bash
ollama create driving-mentor -f Modelfile
ollama run driving-mentor
```

**메모리 요구사항:** Q3_K_M ~3.3GB → RTX 2060 6GB VRAM 100% GPU 가능